# Linked async sky tile + progressive dynamic spectrum

Experiment UI from branch `async-test` (see `.agents/experiment-async-linked-ui.md`).

Paths below mirror the performance benches:

| Section | What you feel | Same code as |
|---------|---------------|--------------|
| **A** Fixed `(l, m)` progressive | Chunks paint into the dynspec; tap → sky tile | `bench_linked` Approach A + `ZarrPixelReader` |
| **B** Accessor fixed `(l, m)` | One-shot `dynamic_spectrum(l=, m=)` (no progressive UI) | Demo / C-native fixed path |
| **C** RA/Dec track + progressive extract | Track first, then A-tracked fill; tap recenters tile on tracked pixel | `bench_radec_track` A-tracked |
| **D** Accessor RA/Dec | One-shot `dynamic_spectrum(ra=, dec=)` | `bench_radec_track` C-native total |
| **E** SkyWidget overlay | Tap dynspec → `update_slice` on HiPS/WebGL | SourceReview overlay path (no Panel app) |

Reads use a `ThreadPoolExecutor`; Bokeh updates run on the kernel io_loop. Section **E** uses `astrowidget.SkyWidget` + `pipeline_qa_app.bind_sky_widget_dataset` (native ipywidgets display — see AGENTS.md simpler-notebook rules).


In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import panel as pn

warnings.filterwarnings("ignore")

HERE = Path.cwd()
if not (HERE / "linked_ui.py").exists():
    HERE = HERE / "notebooks" / "async_cube_slice"
sys.path.insert(0, str(HERE))

from linked_ui import LinkedAsyncView
from bench_radec_track import _crval_at

pn.extension("bokeh")

In [ ]:
# Same store / spatial target as the benches.
STORE = "/fast/claw/IV_1hMHz_Taper_shflux_coadd.zarr"
TILE = slice(1173, 1564)  # center native 391×391; use slice(0, 391) for /fast/claw/async-test/R*.zarr
PIXEL = 1368  # center of that tile; use 195 for subset copies
MAX_WORKERS = 4  # best for ~1 MB chunks in the benchmark
POL = 0

## A — Progressive fixed `(l, m)` (Approach A)

Start the progressive fill here. The **dynspec plot is shown in section F** (next to
SkyWidget) so Bokeh tap callbacks stay on a single notebook mount — showing it in A
and again in F breaks clicks.

This cell shows status + the small Bokeh sky tile only. Run **F** for the clickable
dynspec + FOV overlay.

In [ ]:
view = LinkedAsyncView(
    STORE,
    l_slice=TILE,
    m_slice=TILE,
    l_idx=PIXEL,
    m_idx=PIXEL,
    pol_idx=POL,
    max_workers=MAX_WORKERS,
)
view.layout

In [ ]:
# Same extract path as bench_linked Approach A (ZarrPixelReader + progressive_fill units).
view.start(t=0, f=0)

In [ ]:
view.log

## B — Accessor fixed `(l, m)` (one-shot, for feel / timing)

Mirrors `open_dataset(chunks={})` + `radport.dynamic_spectrum(l=, m=)` from the accessor hot-path bench. Nothing paints until the full array returns.

In [ ]:
import ovro_lwa_portal as ovro

t0 = time.perf_counter()
ds = ovro.open_dataset(STORE, chunks={}, validate=False)
print(f"open_dataset(chunks={{}}) in {time.perf_counter() - t0:.2f} s")

l_val = float(ds.coords["l"].values[PIXEL])
m_val = float(ds.coords["m"].values[PIXEL])
print(f"fixed pixel idx=({PIXEL},{PIXEL}) → l={l_val:+.4f}, m={m_val:+.4f}")

t0 = time.perf_counter()
da_fixed = ds.radport.dynamic_spectrum(l=l_val, m=m_val, pol=POL)
grid_fixed = np.asarray(da_fixed.values, dtype=np.float32)
print(
    f"dynamic_spectrum(l, m) in {time.perf_counter() - t0:.3f} s | "
    f"finite={np.isfinite(grid_fixed).mean():.3f} sum={np.nansum(grid_fixed):.4g}"
)

## C — RA/Dec track + progressive extract (Approach A-tracked)

Same stages as `bench_radec_track.py`:

1. `_compute_pixel_track(ra, dec)` (WCS — typically ~2.5 s on this store)
2. `view.start_tracked(...)` → `ZarrTrackedPixelReader` + progressive paint

Default catalog = phase center at `time_idx=0` (CRVAL). Tap a dynspec cell to load that `(t, f)` sky tile **centered on the tracked pixel** at that time.

In [ ]:
# Catalog: t0 phase center (override RA/DEC to try an offset source).
RA, DEC = _crval_at(ds, time_idx=0)
# RA, DEC = 23.0, 37.0  # optional offset used in the bench

t0 = time.perf_counter()
l_idx, m_idx, visible = ds.radport._compute_pixel_track(RA, DEC)
t_track = time.perf_counter() - t0

n_vis = int(np.count_nonzero(visible))
unique_lm = len({(int(l_idx[t]), int(m_idx[t])) for t in range(len(visible)) if visible[t]})
print(
    f"track RA={RA:.4f} Dec={DEC:.4f} in {t_track:.3f} s | "
    f"visible={n_vis}/{len(visible)} unique_lm={unique_lm}"
)

In [ ]:
# Reuse the same LinkedAsyncView (or create a fresh one).
# Progressive extract = same ZarrTrackedPixelReader path as the RA/Dec bench.
view.start_tracked(
    l_idx,
    m_idx,
    visible,
    title=f"RA={RA:.2f}, Dec={DEC:.2f}",
)
view.layout  # re-display if the previous output scrolled away

In [ ]:
view.log[-8:]

## D — Accessor RA/Dec (one-shot total)

Full `dynamic_spectrum(ra=, dec=)` = track + extract in one call (C-native total in the bench). Compare wall time and grid to the progressive result in **C**.

In [ ]:
t0 = time.perf_counter()
da_radec = ds.radport.dynamic_spectrum(ra=RA, dec=DEC, pol=POL)
grid_radec = np.asarray(da_radec.values, dtype=np.float32)
t_total = time.perf_counter() - t0
print(
    f"dynamic_spectrum(ra, dec) in {t_total:.3f} s | "
    f"finite={np.isfinite(grid_radec).mean():.3f} sum={np.nansum(grid_radec):.4g}"
)

# Parity vs progressive grid from section C (after fill finished).
if np.isfinite(view.grid).any():
    match = np.allclose(
        np.nan_to_num(view.grid), np.nan_to_num(grid_radec), rtol=1e-4, atol=1e-3, equal_nan=True
    )
    print(f"progressive grid vs accessor: close={match} "
          f"|Δsum|={abs(float(np.nansum(view.grid) - np.nansum(grid_radec))):.4g}")
else:
    print("progressive grid still empty — wait for section C to finish, then re-run this cell")

## E — SkyWidget overlay (async dynspec + HiPS/WebGL)

Keeps the progressive dynspec from **A** or **C**. The sky image for the selected
`(t, f)` is shown as an **astrowidget** radio overlay on a HiPS background
(same display stack as SourceReview, without the Panel app).

**Pattern (AGENTS.md simpler notebooks):** create and display `SkyWidget` in **one**
cell via native ipywidgets; bind with `defer_display=True`; drive slices with
`update_slice` from `view.on_slice` (fires on start and on dynspec tap).

Run **after** `ds` exists (section **B**) and `view` is started (**A** or **C**).
Restart the kernel if `astrowidget` / portal code changed under you.


In [ ]:
from astropy import units as u
from astropy.coordinates import SkyCoord
from astrowidget import SkyWidget
from IPython.display import display

from ovro_lwa_portal.viz.pipeline_qa_app import bind_sky_widget_dataset

SKY_FOV_DEG = 8.0  # same default as source_review.ipynb
SKY_MAX_SIZE = 2048  # display stride; Zarr read still follows on-disk chunks

# Prefer catalog center when section C set RA/DEC; else fixed-pixel sky coords.
try:
    _ra, _dec = float(RA), float(DEC)
except NameError:
    _ra, _dec = ds.radport.pixel_to_coords(PIXEL, PIXEL, time_idx=0)
    _ra, _dec = float(_ra), float(_dec)

sky_center = SkyCoord(_ra, _dec, unit="deg", frame="fk5")

sky = SkyWidget()
bind_sky_widget_dataset(sky, ds, max_size=SKY_MAX_SIZE)
sky.background_survey = "DSS"  # HiPS under the radio overlay

def _overlay_slice(t: int, f: int) -> None:
    """Same slice indices as LinkedAsyncView tap / start (bench sky pane)."""
    t0 = time.perf_counter()
    # center= forces catalog/field reproject (SourceReview Center / LPT path).
    # Heatmap-style browsing could use view_lock=True instead — see note below.
    sky.update_slice(
        int(t),
        int(f),
        center=sky_center,
        fov=SKY_FOV_DEG * u.deg,
    )
    if hasattr(sky, "set_crosshair"):
        sky.set_crosshair(sky_center)
    if hasattr(sky, "send_state"):
        sky.send_state()
    view.log.append(f"sky overlay t={t} f={f} in {time.perf_counter() - t0:.3f} s")

view.on_tf_only = None
view.on_slice = _overlay_slice
display(sky)  # same cell as construction — required for widget comm


In [ ]:
# Push the current selection onto the overlay (first visible / last tap).
# Dynspec taps will keep calling view.on_slice afterward.
t0 = int(view.sel_src.data["x"][0] - 0.5)
f0 = int(view.sel_src.data["y"][0] - 0.5)
_overlay_slice(t0, f0)
print(f"overlay at t={t0} f={f0}; tap the dynspec in section A/C to change slices")


**Feel vs Bokeh tile:** the left Bokeh pane is a raw numpy tile (fast, no WCS/HiPS).
SkyWidget adds per-time WCS, optional reproject, and HiPS — expect higher latency per
tap (same class of cost as SourceReview overlay). Progressive dynspec fill is unchanged.

For pan/zoom-preserving taps (heatmap style), change `_overlay_slice` to
`sky.update_slice(t, f, view_lock=True)` with no `center`/`fov`.


## F — FOV-native SkyWidget overlay (Approach 1)

Same progressive dynspec as **A**/**C**, but the radio overlay reads a **FOV-sized
native-resolution cutout** (chunk-aligned) instead of a full-plane `SKY_MAX_SIZE`
stride.

**Layout:** SkyWidget + dynspec are shown together below. This is the **first and only**
mount of `view.dyn_pane` (do not display it in section A).

**Heatmap click** changes only the overlay `(time, freq)`. The red box moves, the
overlay fades, then one async cutout loads at the **current** view. The view, crosshair,
and heatmap values stay put. Log: `dynspec tap`, `slice-fade`, then `FOV overlay [slice]`.

**Heatmap at crosshair** changes only the heatmap values, at the crosshair you
placed on the sky (a click, or `set_crosshair`). It does not use the view center.
The pixel track runs on a worker, then `start_tracked(..., sync_slice=False)` refills
the spectrum. The overlay and the selected `(time, freq)` do not change. With no
crosshair, the button logs that and does not start a fill.

**Pan / zoom:** the shader keeps sampling the current texture. A zoom or pan that stays
inside that cutout does **not** reproject or re-read Zarr (`[pan-skip]` once). A **pan** that no longer
fits in the cache fades the overlay (`[pan-fade]`) until the new cutout arrives. A zoom
that keeps the same center stays at full opacity. Leaving the cache waits briefly, then one
async read (`[pan]`). An empty read is dropped so the overlay already on screen stays up.

**Requires:** `view` started (**A** or **C**), `ds` open (**B**). Section **E** remains
the full-plane baseline.

Headless bench: `pixi run python notebooks/async_cube_slice/bench_fov_cutout.py --quick`

In [ ]:
import panel as pn
import zarr
import ipywidgets as ipw
from astropy.coordinates import SkyCoord
from astrowidget import SkyWidget
from IPython.display import display

from fov_cutout import FovOverlaySession, attach_fov_pan_reload, crosshair_skycoord
from linked_ui import jupyter_schedule_main
from ovro_lwa_portal.viz.pipeline_qa_app import bind_sky_widget_dataset

SKY_FOV_DEG = 8.0
sky_fov = SkyWidget()
bind_sky_widget_dataset(sky_fov, ds, max_size=256)
sky_fov.background_survey = "DSS"

_sky_zarr = zarr.open_consolidated(STORE, mode="r")["SKY"]

try:
    _ra_f, _dec_f = float(RA), float(DEC)
except NameError:
    _ra_f, _dec_f = ds.radport.pixel_to_coords(PIXEL, PIXEL, time_idx=0)
    _ra_f, _dec_f = float(_ra_f), float(_dec_f)

sky_center_fov = SkyCoord(_ra_f, _dec_f, unit="deg", frame="fk5")

fov_session = FovOverlaySession(
    sky_fov,
    ds,
    _sky_zarr,
    pol_idx=POL,
    executor=view.pool,
    pan_pad_frac=0.5,  # working-set margin for small pans
    schedule_main=jupyter_schedule_main,
    log=view.log.append,
)

def _current_sky_view() -> tuple[SkyCoord, float]:
    """Center and FOV the SkyWidget is showing right now."""
    center = sky_fov.view_center_skycoord()
    fov = float(sky_fov.view_fov)
    if fov != fov or fov <= 0:
        fov = SKY_FOV_DEG
    return center, fov


def _overlay_fov_slice(t: int, f: int) -> None:
    """Dynspec tap: new (t, f) cutout at the current view. Does not recenter."""
    center, fov = _current_sky_view()
    fov_session.push_slice(center, fov, t=t, f=f, update_view=False, reason="slice")


heatmap_at_crosshair = pn.widgets.Button(
    name="Heatmap at crosshair", button_type="primary", width=180
)


def _on_heatmap_at_crosshair(event=None) -> None:
    """On demand: async pixel track + progressive fill at the placed crosshair."""
    center = crosshair_skycoord(sky_fov)
    if center is None:
        view.log.append("heatmap: click the sky to place a crosshair first")
        return
    ra = float(center.icrs.ra.deg)
    dec = float(center.icrs.dec.deg)
    heatmap_at_crosshair.disabled = True
    heatmap_at_crosshair.name = "Tracking…"
    view.log.append(f"heatmap at crosshair ({ra:.3f}, {dec:.3f}) — tracking")

    def _ready() -> None:
        heatmap_at_crosshair.disabled = False
        heatmap_at_crosshair.name = "Heatmap at crosshair"
        view.log.append(f"heatmap fill started at ({ra:.3f}, {dec:.3f})")

    def _failed(exc: BaseException) -> None:
        heatmap_at_crosshair.disabled = False
        heatmap_at_crosshair.name = "Heatmap at crosshair"
        view.log.append(f"heatmap track failed: {exc}")

    view.schedule_tracked_fill(
        ra,
        dec,
        ds.radport._compute_pixel_track,
        title=f"crosshair {ra:.2f},{dec:.2f}",
        on_started=_ready,
        on_error=_failed,
    )


heatmap_at_crosshair.on_click(_on_heatmap_at_crosshair)

def _on_fov_pan(center, fov_deg: float) -> None:
    """Gesture end: keep the texture when the cache already covers the view."""
    jupyter_schedule_main(lambda: fov_session.on_pan(center, fov_deg))

def _on_heatmap_tf(t: int, f: int) -> None:
    """Heatmap click: fade, then load this (t, f) at the current view."""
    fov_session.reload_time_freq(t, f)


attach_fov_pan_reload(sky_fov, _on_fov_pan)
view.on_slice = None  # tile paints must not move the overlay
view.on_tf_only = _on_heatmap_tf

# Open on the spectrum coordinate. Later taps and pans keep whatever view is current.
sky_fov.view_ra = float(sky_center_fov.icrs.ra.deg)
sky_fov.view_dec = float(sky_center_fov.icrs.dec.deg)
sky_fov.view_fov = SKY_FOV_DEG

view.dyn_fig.width = 640
view.dyn_fig.height = 420

display(ipw.VBox([
    ipw.HTML(
        "<b>FOV SkyWidget</b> — a dynspec click fades, then changes only time/freq · "
        "Heatmap at crosshair refills the spectrum at the marker, not the view center"
    ),
    sky_fov,
]))
view.refresh_dynspec_display()
pn.Row(
    pn.Column(
        heatmap_at_crosshair,
        pn.pane.Markdown("Refill at the **crosshair**, not the view center.", width=180),
        width=190,
    ),
    view.dyn_pane,
)

In [ ]:
# First cutout at the current view (the spectrum coordinate set above).
# Dynspec taps call the same helper and do not move the view.
t0 = int(view.sel_src.data["x"][0] - 0.5)
f0 = int(view.sel_src.data["y"][0] - 0.5)
_overlay_fov_slice(t0, f0)
print(f"FOV overlay at t={t0} f={f0}; click the dynspec to change slices in place")
print("latest:", view.log[-1] if view.log else "(empty)")

In [ ]:
view.log  # expect "dynspec tap → …" then "FOV overlay [slice] …" after clicks

**Feel vs section E:** E strides a full plane (`SKY_MAX_SIZE`); F reads an FOV-native cutout.

**Try:** pan the SkyWidget, then click the dynspec. The overlay fades, then shows that
cell's time and frequency at the same view. The spectrum values and the view do not change.

Click the sky to place the crosshair, then press **Heatmap at crosshair**. The button
shows Tracking… while the pixel track runs, then the spectrum refills at that marker
(`heatmap at crosshair`, then `dynspec complete`). Panning first does not change the
target. The red box and the overlay stay on the cell you had selected.

Zoom in on the SkyWidget — the overlay should stay bright and in place (one `[pan-skip]`).
Pan off the radio field — the overlay fades (`[pan-fade]`) until one `[pan]` replaces it.
Zoom out while the center is still on the data — it stays bright, then one `[pan]` extends it.

**If taps still do nothing:** Kernel → Restart; section A must only show `view.layout`, not
`view.dyn_pane`.

## Cleanup

In [ ]:
view.on_slice = None
for _wname in ("sky", "sky_fov"):
    try:
        globals()[_wname].clear_image()
    except (NameError, AttributeError):
        pass
view.close()
